# Halo Gas IC Comparison: FMM vs MG

This notebook compares gas stability diagnostics for two isolated-halo runs (MG and FMM).
It evaluates each AMR leaf cell against the halo IC equations from `patch/init/halo/condinit.f90` at that same cell center.

In [1]:
from __future__ import annotations

import importlib.util
import shutil
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Video, display


In [2]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start, *start.parents]:
        if (cand / "utils/py/miniramses.py").exists():
            return cand
    raise FileNotFoundError("Could not locate repository root from current working directory.")

repo_root = find_repo_root(Path.cwd())
module_path = repo_root / "analyze/halo/gas_ic_compare.py"
if not module_path.exists():
    raise FileNotFoundError(f"Cannot find {module_path}")

spec = importlib.util.spec_from_file_location("halo_gas_ic_compare", module_path)
halo = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = halo
spec.loader.exec_module(halo)

print(f"repo_root={repo_root}")
print(f"Loaded module: {module_path}")

repo_root=/home/jl4415/mini-ramses
Loaded module: /home/jl4415/mini-ramses/analyze/halo/gas_ic_compare.py


In [3]:
# -----------------------------
# Top-level paths (edit these)
# -----------------------------
# Point each path to a run root containing output_XXXXX directories.
# Example:
#   MG_RUN_DIR = Path('/path/to/halo_mg')
#   FMM_RUN_DIR = Path('/path/to/halo_fmm')

MG_RUN_DIR =  Path("/home/jl4415/mini-ramses/halo_test/halo_mg/").expanduser()
FMM_RUN_DIR =  Path("/home/jl4415/mini-ramses/halo_test/halo_fmm/").expanduser()

RUNS = {
    "MG": Path(MG_RUN_DIR),
    "FMM": Path(FMM_RUN_DIR),
}

# Optional namelist override per solver (set to Path(...) if needed)
NAMELIST_BY_SOLVER = {
    "MG": None,
    "FMM": None,
}

# Shared analysis settings
HYDRO_PREFIX = "hydro"
NOUT_SELECTION = list(range(2, 21))  # 2..20 inclusive
ALIGN = True
SHIFT_TRIALS = [0.0, -0.5, 0.5, -1.0, 1.0]
RADIAL_BINS = 80
RADIAL_RMAX = None
PROFILE_NOUT = "last"
PRESSURE_GRID_SIZE = 20000

OUTPUT_DIR = repo_root / "tmp/halo_notebook_compare"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Movie settings
MAKE_HYDRO_MOVIE = True
MOVIE_FIELD = "density"      # density, pressure, speed, vr, vx, vy, vz
MOVIE_NXY = 300
MOVIE_FPS = 4
MOVIE_DPI = 140
MOVIE_CMAP = "magma"
MOVIE_WEIGHT = "mass"        # mass or volume (for velocity-like fields)
MOVIE_COMPARE_MODE = "raw"   # raw, delta_to_ic, relative_to_ic
MOVIE_REL_DENOM_FLOOR = 1e-30
MOVIE_VMIN = None             # set number to override auto scaling
MOVIE_VMAX = None             # set number to override auto scaling
MOVIE_VMIN_PERCENTILE = 5.0
MOVIE_VMAX_PERCENTILE = 99.0
MOVIE_SYMMETRIC_SIGNED = True # force +/- symmetric limits for signed fields
MOVIE_SUFFIX = "mp4"         # gif output
MOVIE_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")  # avoids stale notebook GIF cache
MOVIE_LOG_DENSITY = True
FORCE_MP4 = True
NORMALIZE_COORDS_TO_BOX = True
NORMALIZE_MODE = "minmax"   # minmax or modulo
HALO_MOVIE_USE_ANALYSIS_SHIFT = True
HALO_MOVIE_FORCE_MINMAX = True
HALO_MOVIE_PERIODIC_TILE = True

RUNS


{'MG': PosixPath('/home/jl4415/mini-ramses/halo_test/halo_mg'),
 'FMM': PosixPath('/home/jl4415/mini-ramses/halo_test/halo_fmm')}

In [4]:
def select_nouts(discovered: list[int], single_nout: list[int] | None, nout_selection):
    if nout_selection is not None:
        nouts = sorted(int(n) for n in nout_selection)
    elif single_nout is not None:
        nouts = single_nout
    else:
        nouts = discovered
    return nouts


def _parse_fortran_float(text: str) -> float:
    return float(text.strip().replace("D", "E").replace("d", "e"))


def _extract_namelist_block(text: str, block_name: str):
    import re
    m = re.search(rf"(?is)&\s*{block_name}\b(.*?)/", text)
    return None if m is None else m.group(1)


def parse_boxlen_hint_from_namelist(namelist_path: Path | None):
    if namelist_path is None or (not namelist_path.exists()):
        return None
    txt = namelist_path.read_text()

    amr_block = _extract_namelist_block(txt, "AMR_PARAMS")
    if amr_block is not None:
        for line in amr_block.splitlines():
            s = line.split("!")[0].strip()
            if not s or "=" not in s:
                continue
            k, v = s.split("=", 1)
            if k.strip().lower() == "boxlen":
                try:
                    return _parse_fortran_float(v.split(",")[0])
                except Exception:
                    pass

    bound_block = _extract_namelist_block(txt, "BOUNDARY_PARAMS")
    if bound_block is not None:
        for line in bound_block.splitlines():
            s = line.split("!")[0].strip()
            if not s or "=" not in s:
                continue
            k, v = s.split("=", 1)
            if k.strip().lower() == "box_size":
                try:
                    return _parse_fortran_float(v.split(",")[0])
                except Exception:
                    pass

    return None


def detect_coordinate_frame(run_root: Path, nout: int, hydro_prefix: str, info_boxlen: float, physical_boxlen: float):
    cells = halo.load_leaf_hydro_cells(nout=nout, run_root=run_root, hydro_prefix=hydro_prefix)
    mins = np.nanmin(cells.x, axis=1)
    maxs = np.nanmax(cells.x, axis=1)
    meds = np.nanmedian(cells.x, axis=1)
    spans = maxs - mins

    coord_shift = np.zeros(3, dtype=np.float64)
    label = "native"

    # Detect "half-box shifted" coordinates, e.g. 300..900 while physical box is 600.
    # This case appears when info.boxlen is 1200 but physical domain is 600.
    if info_boxlen > 1.5 * physical_boxlen:
        half_phys = 0.5 * physical_boxlen
        err_no_shift = np.sum(np.abs(meds - 0.5 * physical_boxlen))
        err_shifted = np.sum(np.abs((meds - half_phys) - 0.5 * physical_boxlen))
        if err_shifted + 1e-12 < err_no_shift:
            coord_shift[:] = -half_phys
            label = "shifted_half_box"
        else:
            label = "half_box_native"

    # Shift-trial enhancement: add quarter-box shifts when physical/info box lengths differ.
    trial_set = set(float(s) for s in SHIFT_TRIALS)
    if info_boxlen > 0:
        q = 0.5 * (physical_boxlen / info_boxlen)
        if q > 0:
            trial_set.update([q, -q, 3.0 * q, -3.0 * q])
    shift_trials_local = sorted(trial_set)

    return {
        "label": label,
        "coord_shift": coord_shift,
        "mins": mins,
        "maxs": maxs,
        "spans": spans,
        "shift_trials_local": shift_trials_local,
    }




def _normalize_axis_to_box(a: np.ndarray, box_size: float, mode: str = "minmax"):
    arr = np.asarray(a, dtype=np.float64)
    if mode == "modulo":
        return np.mod(arr, box_size)
    amin = np.nanmin(arr)
    amax = np.nanmax(arr)
    span = amax - amin
    if not np.isfinite(span) or span <= 0:
        return np.full_like(arr, 0.5 * box_size)
    # Prefer pure shift when span is already ~box_size (e.g. 300..900 -> 0..600).
    if 0.9 * box_size <= span <= 1.1 * box_size:
        out = arr - amin
    else:
        out = (arr - amin) * (box_size / span)
    return np.clip(out, 0.0, box_size)


def _normalize_xyz_to_box(x: np.ndarray, y: np.ndarray, z: np.ndarray | None, box_size: float, mode: str = "minmax"):
    xn = _normalize_axis_to_box(x, box_size, mode=mode)
    yn = _normalize_axis_to_box(y, box_size, mode=mode)
    if z is None:
        return xn, yn, None
    zn = _normalize_axis_to_box(z, box_size, mode=mode)
    return xn, yn, zn

def _get_field_arrays(cells, fmap, field: str, center_xyz=None):
    rho = cells.u[fmap.rho].astype(np.float64)
    vx = cells.u[fmap.vx].astype(np.float64)
    vy = cells.u[fmap.vy].astype(np.float64)
    vz = cells.u[fmap.vz].astype(np.float64)
    p = cells.u[fmap.pressure].astype(np.float64)
    mass_w = rho * cells.dx**3
    vol_w = cells.dx**3
    vel_w = mass_w if str(MOVIE_WEIGHT).lower() == "mass" else vol_w

    if center_xyz is None:
        xc = 0.5 * (np.nanmax(cells.x[0]) + np.nanmin(cells.x[0]))
        yc = 0.5 * (np.nanmax(cells.x[1]) + np.nanmin(cells.x[1]))
        zc = 0.5 * (np.nanmax(cells.x[2]) + np.nanmin(cells.x[2]))
    else:
        c = np.asarray(center_xyz, dtype=np.float64)
        xc, yc, zc = float(c[0]), float(c[1]), float(c[2])
    dxr = cells.x[0].astype(np.float64) - xc
    dyr = cells.x[1].astype(np.float64) - yc
    dzr = cells.x[2].astype(np.float64) - zc
    rr = np.sqrt(dxr**2 + dyr**2 + dzr**2)
    vr = (dxr * vx + dyr * vy + dzr * vz) / np.maximum(rr, 1e-12)

    if field == "density":
        return rho, mass_w
    if field == "pressure":
        return p, mass_w
    if field == "speed":
        speed = np.sqrt(vx**2 + vy**2 + vz**2)
        return speed, vel_w
    if field == "vx":
        return vx, vel_w
    if field == "vy":
        return vy, vel_w
    if field == "vz":
        return vz, vel_w
    if field == "vr":
        return vr, vel_w
    raise ValueError(f"Unsupported MOVIE_FIELD='{field}'")


def _get_ic_field_array(ref: dict, field: str):
    if field == "density":
        return np.asarray(ref["rho"], dtype=np.float64)
    if field == "pressure":
        return np.asarray(ref["pressure"], dtype=np.float64)
    if field == "vx":
        return np.asarray(ref["vx"], dtype=np.float64)
    if field == "vy":
        return np.asarray(ref["vy"], dtype=np.float64)
    if field == "vz":
        return np.asarray(ref["vz"], dtype=np.float64)
    if field == "speed":
        vx = np.asarray(ref["vx"], dtype=np.float64)
        vy = np.asarray(ref["vy"], dtype=np.float64)
        vz = np.asarray(ref["vz"], dtype=np.float64)
        return np.sqrt(vx**2 + vy**2 + vz**2)
    if field == "vr":
        xx = np.asarray(ref["xx"], dtype=np.float64)
        yy = np.asarray(ref["yy"], dtype=np.float64)
        zz = np.asarray(ref["zz"], dtype=np.float64)
        vx = np.asarray(ref["vx"], dtype=np.float64)
        vy = np.asarray(ref["vy"], dtype=np.float64)
        vz = np.asarray(ref["vz"], dtype=np.float64)
        rr = np.sqrt(xx**2 + yy**2 + zz**2)
        return (xx * vx + yy * vy + zz * vz) / np.maximum(rr, 1e-12)
    raise ValueError(f"Unsupported MOVIE_FIELD='{field}' for IC reference")




def _resize_nearest(img: np.ndarray, nx: int, ny: int):
    h, w = img.shape
    if h == ny and w == nx:
        return img
    iy = np.linspace(0, h - 1, ny).astype(int)
    ix = np.linspace(0, w - 1, nx).astype(int)
    return img[np.ix_(iy, ix)]



def _compute_solver_bounds(run_root: Path, nouts: list[int], hydro_prefix: str, summary_df: pd.DataFrame | None, coord_shift):
    if not nouts:
        raise RuntimeError("No nouts provided for bounds computation.")
    n0 = int(nouts[0])
    cells = halo.load_leaf_hydro_cells(nout=n0, run_root=run_root, hydro_prefix=hydro_prefix)
    sh = _lookup_shift_for_nout(summary_df if HALO_MOVIE_USE_ANALYSIS_SHIFT else None, n0, coord_shift)
    x = cells.x[0] + sh[0]
    y = cells.x[1] + sh[1]
    return {
        "xmin": float(np.nanmin(x)),
        "xmax": float(np.nanmax(x)),
        "ymin": float(np.nanmin(y)),
        "ymax": float(np.nanmax(y)),
    }


def _merge_bounds(b1: dict, b2: dict):
    return {
        "xmin": min(b1["xmin"], b2["xmin"]),
        "xmax": max(b1["xmax"], b2["xmax"]),
        "ymin": min(b1["ymin"], b2["ymin"]),
        "ymax": max(b1["ymax"], b2["ymax"]),
    }

def _render_xy_map(cells, fmap, field: str, nxy: int, plot_boxlen: float, coord_shift=None, fixed_bounds: dict | None = None, model=None, analytic_center=None, compare_mode: str = "raw"):
    shift = np.zeros(3) if coord_shift is None else np.asarray(coord_shift)
    x_raw = cells.x[0] + shift[0]
    y_raw = cells.x[1] + shift[1]
    z_raw = cells.x[2] + shift[2]
    dx_raw = cells.dx.astype(np.float64)

    if fixed_bounds is not None:
        xmin = fixed_bounds["xmin"]
        xmax = fixed_bounds["xmax"]
        ymin = fixed_bounds["ymin"]
        ymax = fixed_bounds["ymax"]
        span = max(xmax - xmin, ymax - ymin, 1e-30)
        x = (x_raw - xmin) * (plot_boxlen / span)
        y = (y_raw - ymin) * (plot_boxlen / span)
        z = z_raw
        dx = dx_raw * (plot_boxlen / span)
    elif NORMALIZE_COORDS_TO_BOX:
        norm_mode = "minmax" if HALO_MOVIE_FORCE_MINMAX else NORMALIZE_MODE
        x, y, z = _normalize_xyz_to_box(x_raw, y_raw, z_raw, box_size=plot_boxlen, mode=norm_mode)
        span_x = max(np.nanmax(x_raw) - np.nanmin(x_raw), 1e-30)
        scale = plot_boxlen / span_x
        dx = dx_raw * scale
    else:
        x, y, z = x_raw, y_raw, z_raw
        dx = dx_raw

    # Hard safety net
    outside = np.mean((x < 0.0) | (x > plot_boxlen) | (y < 0.0) | (y > plot_boxlen))
    if outside > 1e-6:
        x, y, z = _normalize_xyz_to_box(x, y, z, box_size=plot_boxlen, mode="minmax")
        span_x = max(np.nanmax(x_raw) - np.nanmin(x_raw), 1e-30)
        scale = plot_boxlen / span_x
        dx = dx_raw * scale

    sim_values, mass = _get_field_arrays(cells, fmap, field, center_xyz=analytic_center)

    mode = str(compare_mode).lower().strip()
    if mode == "raw":
        values = sim_values
    else:
        if model is None:
            raise RuntimeError(f"MOVIE_COMPARE_MODE='{compare_mode}' requires halo model.")
        ref = model.evaluate(cells.x[0], cells.x[1], cells.x[2], center_override=analytic_center)
        ref_values = _get_ic_field_array(ref, field)
        if mode == "delta_to_ic":
            values = sim_values - ref_values
        elif mode == "relative_to_ic":
            denom = np.maximum(np.abs(ref_values), float(MOVIE_REL_DENOM_FLOOR))
            values = (sim_values - ref_values) / denom
        else:
            raise ValueError(f"Unsupported MOVIE_COMPARE_MODE='{compare_mode}'")

    periodic_tiled = False
    num_w = values * mass
    den_w = mass

    if HALO_MOVIE_PERIODIC_TILE:
        shifts = (-plot_boxlen, 0.0, plot_boxlen)
        x_tiles = []
        y_tiles = []
        dx_tiles = []
        num_tiles = []
        den_tiles = []
        for sx in shifts:
            for sy in shifts:
                x_tiles.append(x + sx)
                y_tiles.append(y + sy)
                dx_tiles.append(dx)
                num_tiles.append(num_w)
                den_tiles.append(den_w)

        x_img = np.concatenate(x_tiles)
        y_img = np.concatenate(y_tiles)
        dx_img = np.concatenate(dx_tiles)
        num_img = np.concatenate(num_tiles)
        den_img = np.concatenate(den_tiles)

        num_big = halo.ram.mk_image(x_img, y_img, dx_img, num_img)
        den_big = halo.ram.mk_image(x_img, y_img, dx_img, den_img)
        map_big = num_big / np.maximum(den_big, 1e-30)

        ny_big, nx_big = map_big.shape
        ix0, ix1 = nx_big // 3, 2 * (nx_big // 3)
        iy0, iy1 = ny_big // 3, 2 * (ny_big // 3)
        map2d = map_big[iy0:iy1, ix0:ix1]
        periodic_tiled = True
    else:
        num_map = halo.ram.mk_image(x, y, dx, num_w)
        den_map = halo.ram.mk_image(x, y, dx, den_w)
        map2d = num_map / np.maximum(den_map, 1e-30)
    map2d = _resize_nearest(map2d, nxy, nxy)

    if (field == "density") and MOVIE_LOG_DENSITY and (mode == "raw"):
        map2d = np.log10(np.maximum(map2d, 1e-30))

    meta = {
        "raw_xmin": float(np.nanmin(x_raw)),
        "raw_xmax": float(np.nanmax(x_raw)),
        "raw_ymin": float(np.nanmin(y_raw)),
        "raw_ymax": float(np.nanmax(y_raw)),
        "norm_xmin": float(np.nanmin(x)),
        "norm_xmax": float(np.nanmax(x)),
        "norm_ymin": float(np.nanmin(y)),
        "norm_ymax": float(np.nanmax(y)),
        "outside": float(outside),
        "periodic_tiled": bool(periodic_tiled),
        "compare_mode": mode,
    }
    return map2d, meta


def _lookup_shift_for_nout(summary_df: pd.DataFrame | None, nout: int, default_shift):
    base = np.zeros(3, dtype=np.float64) if default_shift is None else np.asarray(default_shift, dtype=np.float64)
    if summary_df is None:
        return base
    if not {"nout", "shift_x", "shift_y", "shift_z"}.issubset(summary_df.columns):
        return base
    rows = summary_df[summary_df["nout"].astype(int) == int(nout)]
    if rows.empty:
        return base
    return np.array([
        float(rows["shift_x"].iloc[0]),
        float(rows["shift_y"].iloc[0]),
        float(rows["shift_z"].iloc[0]),
    ], dtype=np.float64)


def _lookup_analytic_center_for_nout(summary_df: pd.DataFrame | None, nout: int, default_center):
    if default_center is None:
        return None
    c0 = np.asarray(default_center, dtype=np.float64)
    if summary_df is None:
        return c0
    req = {"nout", "analytic_center_x", "analytic_center_y", "analytic_center_z"}
    if not req.issubset(summary_df.columns):
        return c0
    rows = summary_df[summary_df["nout"].astype(int) == int(nout)]
    if rows.empty:
        return c0
    return np.array([
        float(rows["analytic_center_x"].iloc[0]),
        float(rows["analytic_center_y"].iloc[0]),
        float(rows["analytic_center_z"].iloc[0]),
    ], dtype=np.float64)


def _load_map_sequence(
    run_root: Path,
    nouts: list[int],
    hydro_prefix: str,
    field: str,
    nxy: int,
    plot_boxlen: float,
    coord_shift=None,
    summary_df: pd.DataFrame | None = None,
    fixed_bounds: dict | None = None,
    model=None,
    compare_mode: str = "raw",
):
    maps = []
    times = []
    for i, nout in enumerate(nouts):
        info = halo.ram.rd_info(nout, path=str(run_root))
        cells = halo.load_leaf_hydro_cells(nout=nout, run_root=run_root, hydro_prefix=hydro_prefix)
        names = halo.read_hydro_var_names(run_root / f"output_{nout:05d}", hydro_prefix)
        fmap = halo.build_field_map(cells.nvar, names)

        shift_nout = _lookup_shift_for_nout(summary_df if HALO_MOVIE_USE_ANALYSIS_SHIFT else None, nout, coord_shift)
        analytic_center_nout = _lookup_analytic_center_for_nout(summary_df, nout, None if model is None else model.center)

        if i == 0:
            x_probe = cells.x[0] + shift_nout[0]
            y_probe = cells.x[1] + shift_nout[1]
            outside = float(np.mean((x_probe < 0.0) | (x_probe > plot_boxlen) | (y_probe < 0.0) | (y_probe > plot_boxlen)))
            print(f"[movie] {run_root.name} nout={nout} shift={shift_nout} outside_frac={outside:.3e}")

        m, meta = _render_xy_map(
            cells,
            fmap,
            field=field,
            nxy=nxy,
            plot_boxlen=plot_boxlen,
            coord_shift=shift_nout,
            fixed_bounds=fixed_bounds,
            model=model,
            analytic_center=analytic_center_nout,
            compare_mode=compare_mode,
        )
        if i == 0:
            print(
                f"[movie-bounds] {run_root.name} nout={nout} "
                f"raw_x=[{meta['raw_xmin']:.3f},{meta['raw_xmax']:.3f}] raw_y=[{meta['raw_ymin']:.3f},{meta['raw_ymax']:.3f}] "
                f"norm_x=[{meta['norm_xmin']:.3f},{meta['norm_xmax']:.3f}] norm_y=[{meta['norm_ymin']:.3f},{meta['norm_ymax']:.3f}] "
                f"ptiled={meta['periodic_tiled']}"
            )
        maps.append(m)
        times.append(float(info.time))
    return maps, times, plot_boxlen


def _movie_signed_field(field: str, compare_mode: str) -> bool:
    mode = str(compare_mode).lower().strip()
    if mode in {"delta_to_ic", "relative_to_ic"}:
        return True
    return field in {"vx", "vy", "vz", "vr"}


def _compute_movie_limits(arr: np.ndarray, field: str, compare_mode: str):
    if (MOVIE_VMIN is not None) and (MOVIE_VMAX is not None):
        vmin = float(MOVIE_VMIN)
        vmax = float(MOVIE_VMAX)
    else:
        finite = np.isfinite(arr)
        vals = arr[finite]
        if vals.size == 0:
            vmin, vmax = 0.0, 1.0
        else:
            vmin = float(np.percentile(vals, float(MOVIE_VMIN_PERCENTILE)))
            vmax = float(np.percentile(vals, float(MOVIE_VMAX_PERCENTILE)))
            if vmin == vmax:
                vmax = vmin + 1e-12

    if MOVIE_SYMMETRIC_SIGNED and _movie_signed_field(field, compare_mode):
        amp = max(abs(vmin), abs(vmax), 1e-30)
        vmin, vmax = -amp, amp

    return vmin, vmax


def _save_single_movie(maps, times, boxlen, out_path: Path, title_prefix: str, field: str, compare_mode: str, fps: int = 4, dpi: int = 140):
    arr = np.array(maps)
    if arr.size == 0:
        raise RuntimeError("No maps provided for movie.")

    vmin, vmax = _compute_movie_limits(arr, field, compare_mode)

    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    im = ax.imshow(maps[0], origin="lower", extent=(0, boxlen, 0, boxlen), cmap=MOVIE_CMAP, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(f"projected {field} [{compare_mode}]")
    ttl = ax.set_title(f"{title_prefix} | t={times[0]:.6g}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    def _update(i):
        im.set_data(maps[i])
        ttl.set_text(f"{title_prefix} | t={times[i]:.6g}")
        return [im, ttl]

    ani = animation.FuncAnimation(fig, _update, frames=len(maps), interval=1000 / max(fps, 1), blit=False)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.suffix.lower() == ".mp4":
        if shutil.which("ffmpeg") is None:
            raise RuntimeError("ffmpeg not found for mp4 output. Set MOVIE_SUFFIX='gif'.")
        writer = animation.FFMpegWriter(fps=fps)
    else:
        writer = animation.PillowWriter(fps=fps)
    ani.save(str(out_path), writer=writer, dpi=dpi)
    plt.close(fig)


def _save_comparison_movie(maps_mg, maps_fmm, times, boxlen, out_path: Path, field: str, compare_mode: str, fps: int = 4, dpi: int = 140):
    arr = np.array(list(maps_mg) + list(maps_fmm))
    vmin, vmax = _compute_movie_limits(arr, field, compare_mode)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    im0 = axes[0].imshow(maps_mg[0], origin="lower", extent=(0, boxlen, 0, boxlen), cmap=MOVIE_CMAP, vmin=vmin, vmax=vmax)
    im1 = axes[1].imshow(maps_fmm[0], origin="lower", extent=(0, boxlen, 0, boxlen), cmap=MOVIE_CMAP, vmin=vmin, vmax=vmax)
    axes[0].set_title("MG")
    axes[1].set_title("FMM")
    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.8, label=f"projected {field} [{compare_mode}]")
    st = fig.suptitle(f"Hydro map comparison ({field}, {compare_mode}) | t={times[0]:.6g}")

    def _update(i):
        im0.set_data(maps_mg[i])
        im1.set_data(maps_fmm[i])
        st.set_text(f"Hydro map comparison ({field}, {compare_mode}) | t={times[i]:.6g}")
        return [im0, im1, st]

    ani = animation.FuncAnimation(fig, _update, frames=len(times), interval=1000 / max(fps, 1), blit=False)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.suffix.lower() == ".mp4":
        if shutil.which("ffmpeg") is None:
            raise RuntimeError("ffmpeg not found for mp4 output. Set MOVIE_SUFFIX='gif'.")
        writer = animation.FFMpegWriter(fps=fps)
    else:
        writer = animation.PillowWriter(fps=fps)
    ani.save(str(out_path), writer=writer, dpi=dpi)
    plt.close(fig)


def analyze_solver(label: str, run_dir: Path, namelist_override=None):
    run_root, single_nout = halo.resolve_run_root(Path(run_dir), HYDRO_PREFIX)
    discovered = halo.discover_outputs(run_root, HYDRO_PREFIX)
    nouts = select_nouts(discovered, single_nout, NOUT_SELECTION)
    if not nouts:
        raise RuntimeError(f"[{label}] No outputs selected/discovered under {run_root}")

    first_output_dir = run_root / f"output_{nouts[0]:05d}"
    if namelist_override is not None:
        namelist_path = Path(namelist_override)
    else:
        candidate = first_output_dir / "namelist.txt"
        namelist_path = candidate if candidate.exists() else None

    info0 = halo.ram.rd_info(nouts[0], path=str(run_root))

    boxlen_hint = parse_boxlen_hint_from_namelist(namelist_path)
    info_boxlen = float(info0.boxlen)
    if boxlen_hint is not None and boxlen_hint > 0 and boxlen_hint <= info_boxlen:
        model_boxlen = float(boxlen_hint)
    else:
        model_boxlen = info_boxlen

    frame = detect_coordinate_frame(
        run_root=run_root,
        nout=nouts[0],
        hydro_prefix=HYDRO_PREFIX,
        info_boxlen=info_boxlen,
        physical_boxlen=model_boxlen,
    )

    if namelist_path is not None:
        params = halo.parse_halo_params_from_namelist(namelist_path)
    else:
        params = halo.HaloParams(halo_center=np.zeros(3, dtype=np.float64))

    model = halo.HaloICModel(
        params=params,
        boxlen=model_boxlen,
        unit_d=float(info0.unit_d),
        unit_l=float(info0.unit_l),
        unit_t=float(info0.unit_t),
        pressure_grid_size=PRESSURE_GRID_SIZE,
    )

    outdir = OUTPUT_DIR / label.lower()
    outdir.mkdir(parents=True, exist_ok=True)

    summary_rows = []
    level_dfs = []
    profile_target = halo.pick_profile_nout(nouts, PROFILE_NOUT)
    radial_profile = pd.DataFrame()

    print(f"[{label}] run_root={run_root}")
    print(f"[{label}] nouts={nouts}")
    print(f"[{label}] namelist={namelist_path}")
    print(f"[{label}] info_boxlen={info_boxlen}, model_boxlen={model_boxlen}")
    print(f"[{label}] coord_frame={frame['label']}, mins={frame['mins']}, maxs={frame['maxs']}")
    print(f"[{label}] shift_trials={frame['shift_trials_local']}")

    for nout in nouts:
        summary, radial_df, level_df = halo.analyze_snapshot(
            run_root=run_root,
            nout=nout,
            model=model,
            hydro_prefix=HYDRO_PREFIX,
            align=ALIGN,
            shift_trials=frame['shift_trials_local'],
            radial_bins=RADIAL_BINS,
            radial_rmax=RADIAL_RMAX,
        )
        summary_rows.append(summary)
        if not level_df.empty:
            level_dfs.append(level_df)
        if nout == profile_target:
            radial_profile = radial_df.copy()

    summary_df = pd.DataFrame(summary_rows).sort_values("nout").reset_index(drop=True)
    summary_path = outdir / "gas_ic_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    level_path = None
    if level_dfs:
        level_all = pd.concat(level_dfs, ignore_index=True)
        level_path = outdir / "gas_ic_level_stats.csv"
        level_all.to_csv(level_path, index=False)

    radial_path = None
    radial_png = None
    if not radial_profile.empty:
        radial_path = outdir / f"gas_ic_radial_profile_nout{profile_target:05d}.csv"
        radial_profile.to_csv(radial_path, index=False)

    ts_png = outdir / "gas_ic_timeseries.png"
    halo.plot_timeseries(summary_df, ts_png)

    if not radial_profile.empty:
        radial_png = outdir / f"gas_ic_radial_nout{profile_target:05d}.png"
        halo.plot_radial_profile(radial_profile, radial_png, profile_target)

    return {
        "label": label,
        "run_root": run_root,
        "nouts": nouts,
        "summary": summary_df,
        "summary_path": summary_path,
        "level_path": level_path,
        "radial_path": radial_path,
        "timeseries_png": ts_png,
        "radial_png": radial_png,
        "outdir": outdir,
        "coord_shift": frame['coord_shift'],
        "model": model,
        "frame_label": frame['label'],
        "model_boxlen": model_boxlen,
        "info_boxlen": info_boxlen,
        "shift_trials_local": frame['shift_trials_local'],
    }


def build_comparison(mg: pd.DataFrame, fmm: pd.DataFrame):
    mg = mg.copy().sort_values(["nout", "time"]).reset_index(drop=True)
    fmm = fmm.copy().sort_values(["nout", "time"]).reset_index(drop=True)

    common_nout = sorted(set(mg["nout"].astype(int)).intersection(set(fmm["nout"].astype(int))))
    if common_nout:
        a = mg[mg["nout"].isin(common_nout)].sort_values("nout")
        b = fmm[fmm["nout"].isin(common_nout)].sort_values("nout")
        cmp_df = a.merge(b, on="nout", suffixes=("_mg", "_fmm"), how="inner")
        cmp_df["time_ref"] = cmp_df["time_mg"]
        match_mode = "nout"
    else:
        a = mg.sort_values("time")
        b = fmm.sort_values("time")
        cmp_df = pd.merge_asof(a, b, on="time", direction="nearest", suffixes=("_mg", "_fmm"))
        cmp_df = cmp_df.rename(columns={"time": "time_ref"})
        match_mode = "nearest_time"

    metrics = [
        "rms_abs_vr_mass",
        "rms_abs_dvphi_mass",
        "rms_rel_rho_mass",
        "rms_rel_pressure_mass",
        "p99_abs_vr_mass",
        "p99_abs_dvphi_mass",
    ]

    for m in metrics:
        mg_col = f"{m}_mg"
        fmm_col = f"{m}_fmm"
        if mg_col in cmp_df.columns and fmm_col in cmp_df.columns:
            cmp_df[f"delta_{m}"] = cmp_df[fmm_col] - cmp_df[mg_col]
            denom = cmp_df[mg_col].replace(0.0, np.nan)
            cmp_df[f"ratio_{m}_fmm_over_mg"] = cmp_df[fmm_col] / denom

    return cmp_df, match_mode


In [5]:
if RUNS["MG"].resolve() == RUNS["FMM"].resolve():
    raise ValueError("MG_RUN_DIR and FMM_RUN_DIR are identical. Set FMM_RUN_DIR to halo_fmm for isolated-halo comparison.")

results = {}
for label, run_dir in RUNS.items():
    results[label] = analyze_solver(label, run_dir, NAMELIST_BY_SOLVER.get(label))

mg_summary = results["MG"]["summary"]
fmm_summary = results["FMM"]["summary"]

comparison_df, match_mode = build_comparison(mg_summary, fmm_summary)
comparison_path = OUTPUT_DIR / "mg_fmm_gas_ic_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(f"Saved comparison CSV: {comparison_path}")
print(f"Match mode: {match_mode}")

ncpu=1 ndim=3 nlevelmax=13
Time= 0.0516654103659642
Reading grid data...
Found nvar=7
Reading hydro data...
[MG] run_root=/home/jl4415/mini-ramses/halo_test/halo_mg
[MG] nouts=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
[MG] namelist=/home/jl4415/mini-ramses/halo_test/halo_mg/output_00002/namelist.txt
[MG] info_boxlen=1200.0, model_boxlen=600.0
[MG] coord_frame=shifted_half_box, mins=[304.6875 304.6875 304.6875], maxs=[895.3125 895.3125 895.3125]
[MG] shift_trials=[-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]
ncpu=1 ndim=3 nlevelmax=13
Time= 0.0516654103659642
Reading grid data...
Found nvar=7
Reading hydro data...
ncpu=1 ndim=3 nlevelmax=13
Time= 0.10051959079904
Reading grid data...
Found nvar=7
Reading hydro data...
ncpu=1 ndim=3 nlevelmax=13
Time= 0.151471246074238
Reading grid data...
Found nvar=7
Reading hydro data...
ncpu=1 ndim=3 nlevelmax=13
Time= 0.202424902606157
Reading grid data...
Found nvar=7
Reading hydro data...
ncpu=1 ndim=3 nlevelmax=

In [6]:
# --------------------------------------------------
# Overlay plots (MG vs FMM)
# --------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

overlay_metrics = [
    ("rms_abs_vr_mass", r"$\mathrm{RMS}\!\left(|v_r|\right)$"),
    ("rms_abs_dvphi_mass", r"$\mathrm{RMS}\!\left(|v_\phi - v_{\phi,\mathrm{IC}}|\right)$"),
    ("rms_rel_rho_mass", r"$\mathrm{RMS}\!\left(\left|\frac{\Delta \rho}{\rho}\right|\right)$"),
    ("rms_rel_pressure_mass", r"$\mathrm{RMS}\!\left(\left|\frac{\Delta P}{P}\right|\right)$"),
]

for ax, (m, title) in zip(axes.flat, overlay_metrics):
    y_mg = np.abs(mg_summary[m])
    y_fmm = np.abs(fmm_summary[m])

    ax.plot(mg_summary["time"], y_mg, label="MG", lw=2)
    ax.plot(fmm_summary["time"], y_fmm, label="FMM", lw=2, linestyle="--")

    ax.set_title(title)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(title)

    # safer log scaling
    if np.nanmax(y_mg) > 0 and np.nanmax(y_fmm) > 0:
        ax.set_yscale("log")

    ax.grid(alpha=0.25)
    ax.legend()

overlay_png = OUTPUT_DIR / "mg_fmm_overlay.png"
fig.savefig(overlay_png, bbox_inches="tight")
plt.close(fig)

# --------------------------------------------------
# Delta plots (FMM - MG)
# --------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

delta_metrics = [
    ("delta_rms_abs_vr_mass", r"$\Delta\,\mathrm{RMS}\!\left(|v_r|\right)$"),
    ("delta_rms_abs_dvphi_mass", r"$\Delta\,\mathrm{RMS}\!\left(|v_\phi - v_{\phi,\mathrm{IC}}|\right)$"),
    ("delta_rms_rel_rho_mass", r"$\Delta\,\mathrm{RMS}\!\left(\left|\frac{\Delta \rho}{\rho}\right|\right)$"),
    ("delta_rms_rel_pressure_mass", r"$\Delta\,\mathrm{RMS}\!\left(\left|\frac{\Delta P}{P}\right|\right)$"),
]

for ax, (m, title) in zip(axes.flat, delta_metrics):
    if m in comparison_df.columns:
        ax.plot(comparison_df["time_ref"], comparison_df[m], lw=2)

    ax.axhline(0.0, color="black", lw=1, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(title)
    ax.grid(alpha=0.25)

delta_png = OUTPUT_DIR / "mg_fmm_delta.png"
fig.savefig(delta_png, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {overlay_png}")
print(f"Saved: {delta_png}")

Saved: /home/jl4415/mini-ramses/tmp/halo_notebook_compare/mg_fmm_overlay.png
Saved: /home/jl4415/mini-ramses/tmp/halo_notebook_compare/mg_fmm_delta.png


In [7]:
if MAKE_HYDRO_MOVIE:
    RENDERER_VERSION = "2026-04-15-fix-circle-v6-ic-compare-modes"
    print(f"RENDERER_VERSION={RENDERER_VERSION}")
    if FORCE_MP4 and shutil.which("ffmpeg") is None:
        raise RuntimeError("FORCE_MP4=True but ffmpeg is not available. Install ffmpeg or set FORCE_MP4=False.")

    if FORCE_MP4:
        ext = ".mp4"
    else:
        ext = ".mp4" if (MOVIE_SUFFIX.lower() == "mp4" and shutil.which("ffmpeg") is not None) else ".gif"

    movie_paths = {}
    solver_bounds = {}
    other_ext = ".gif" if ext == ".mp4" else ".mp4"
    tag = "" if (MOVIE_TAG is None or str(MOVIE_TAG).strip() == "") else f"_{MOVIE_TAG}"
    print(f"MOVIE_TAG={tag if tag else '(none)'}")

    for label in ["MG", "FMM"]:
        solver_bounds[label] = _compute_solver_bounds(results[label]["run_root"], results[label]["nouts"], HYDRO_PREFIX, results[label]["summary"], results[label]["coord_shift"])
        run_root = results[label]["run_root"]
        nouts = results[label]["nouts"]
        coord_shift = results[label]["coord_shift"]
        plot_boxlen = results[label]["model_boxlen"]
        maps, times, boxlen = _load_map_sequence(
            run_root=run_root,
            nouts=nouts,
            hydro_prefix=HYDRO_PREFIX,
            field=MOVIE_FIELD,
            nxy=MOVIE_NXY,
            plot_boxlen=plot_boxlen,
            coord_shift=coord_shift,
            summary_df=results[label]["summary"],
            fixed_bounds=solver_bounds[label],
            model=results[label]["model"],
            compare_mode=MOVIE_COMPARE_MODE,
        )

        out_path = OUTPUT_DIR / label.lower() / f"{label.lower()}_hydro_{MOVIE_FIELD}_{MOVIE_COMPARE_MODE}_xy{tag}{ext}"
        stale = OUTPUT_DIR / label.lower() / f"{label.lower()}_hydro_{MOVIE_FIELD}_{MOVIE_COMPARE_MODE}_xy{tag}{other_ext}"
        if stale.exists():
            stale.unlink()
        _save_single_movie(maps, times, boxlen, out_path, title_prefix=f"{label} {MOVIE_FIELD}", field=MOVIE_FIELD, compare_mode=MOVIE_COMPARE_MODE, fps=MOVIE_FPS, dpi=MOVIE_DPI)
        movie_paths[f"{label}_single"] = out_path
        print(f"Saved {label} movie: {out_path}")

    common_nouts = sorted(set(results["MG"]["nouts"]).intersection(set(results["FMM"]["nouts"])))
    if common_nouts:
        bounds_cmp = _merge_bounds(solver_bounds["MG"], solver_bounds["FMM"])
        mg_maps, mg_times, mg_box = _load_map_sequence(
            run_root=results["MG"]["run_root"],
            nouts=common_nouts,
            hydro_prefix=HYDRO_PREFIX,
            field=MOVIE_FIELD,
            nxy=MOVIE_NXY,
            plot_boxlen=results["MG"]["model_boxlen"],
            coord_shift=results["MG"]["coord_shift"],
            summary_df=results["MG"]["summary"],
            fixed_bounds=bounds_cmp,
            model=results["MG"]["model"],
            compare_mode=MOVIE_COMPARE_MODE,
        )
        fmm_maps, fmm_times, fmm_box = _load_map_sequence(
            run_root=results["FMM"]["run_root"],
            nouts=common_nouts,
            hydro_prefix=HYDRO_PREFIX,
            field=MOVIE_FIELD,
            nxy=MOVIE_NXY,
            plot_boxlen=results["FMM"]["model_boxlen"],
            coord_shift=results["FMM"]["coord_shift"],
            summary_df=results["FMM"]["summary"],
            fixed_bounds=bounds_cmp,
            model=results["FMM"]["model"],
            compare_mode=MOVIE_COMPARE_MODE,
        )

        boxlen_cmp = min(mg_box, fmm_box)
        cmp_path = OUTPUT_DIR / f"mg_fmm_hydro_{MOVIE_FIELD}_{MOVIE_COMPARE_MODE}_xy_compare{tag}{ext}"
        stale_cmp = OUTPUT_DIR / f"mg_fmm_hydro_{MOVIE_FIELD}_{MOVIE_COMPARE_MODE}_xy_compare{tag}{other_ext}"
        if stale_cmp.exists():
            stale_cmp.unlink()
        _save_comparison_movie(mg_maps, fmm_maps, mg_times, boxlen_cmp, cmp_path, field=MOVIE_FIELD, compare_mode=MOVIE_COMPARE_MODE, fps=MOVIE_FPS, dpi=MOVIE_DPI)
        movie_paths["comparison"] = cmp_path
        print(f"Saved MG/FMM comparison movie: {cmp_path}")
else:
    print("MAKE_HYDRO_MOVIE=False, skipping movie generation.")


RENDERER_VERSION=2026-04-15-fix-circle-v6-ic-compare-modes
MOVIE_TAG=_20260416_090558
ncpu=1 ndim=3 nlevelmax=13
Time= 0.0516654103659642
Reading grid data...
Found nvar=7
Reading hydro data...
ncpu=1 ndim=3 nlevelmax=13
Time= 0.0516654103659642
Reading grid data...
Found nvar=7
Reading hydro data...
[movie] halo_mg nout=2 shift=[-300. -300. -300.] outside_frac=0.000e+00
Making image of size:  12160 12160
Making image of size:  12160 12160
[movie-bounds] halo_mg nout=2 raw_x=[4.688,595.312] raw_y=[4.688,595.312] norm_x=[0.000,600.000] norm_y=[0.000,600.000] ptiled=True
ncpu=1 ndim=3 nlevelmax=13
Time= 0.10051959079904
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.151471246074238
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.202424902606157
Reading grid data...

ncpu=1 ndim=3 nlevelmax=13
Time= 0.151471246074238
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.202424902606157
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.250949929328013
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.301938061759293
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.350304345647971
Reading grid data...
Found nvar=7
Reading hydro data...
Making image of size:  12160 12160
Making image of size:  12160 12160
ncpu=1 ndim=3 nlevelmax=13
Time= 0.400624919488876
Reading grid data...
Found nvar=7
Reading hydro data...
Making i

In [10]:
from IPython.display import Video, Image, display

if MAKE_HYDRO_MOVIE:
    for k, p in movie_paths.items():
        print(k, p)
        if p.suffix.lower() == ".gif":
            print("WARNING: GIF selected; use MP4 to avoid dithering artifacts.")

    if "comparison" in movie_paths and movie_paths["comparison"].exists():
        mpath = movie_paths["comparison"]

        if mpath.suffix.lower() == ".mp4":
            display(Video(str(mpath), embed=True))
        else:
            display(Image(filename=str(mpath)))

MG_single /home/jl4415/mini-ramses/tmp/halo_notebook_compare/mg/mg_hydro_density_raw_xy_20260416_090558.mp4
FMM_single /home/jl4415/mini-ramses/tmp/halo_notebook_compare/fmm/fmm_hydro_density_raw_xy_20260416_090558.mp4
comparison /home/jl4415/mini-ramses/tmp/halo_notebook_compare/mg_fmm_hydro_density_raw_xy_compare_20260416_090558.mp4


In [ ]:
comparison_df.head(20)

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "mg_fmm_overlay.png")))
display(Image(filename=str(OUTPUT_DIR / "mg_fmm_delta.png")))